# Final model: test evaluation, residual analysis, saved pipeline
Selected model: **Linear Regression**

In [ ]:
import sys
sys.path.append("..")

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from src.preprocess import FEATURES, MODEL_PATH, load_data, split_data, build_pipeline, build_cv, summarize_cv

sns.set_style("whitegrid")

df = load_data()
X_train, X_test, y_train, y_test = split_data(df)
cv = build_cv()

final_pipe = build_pipeline(LinearRegression())
_ = summarize_cv(final_pipe, "Linear Regression (validation, for reference)", X_train, y_train, cv)

## Final test evaluation (once)

In [ ]:
final_pipe.fit(X_train, y_train)
y_pred = final_pipe.predict(X_test)

test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_mae = mean_absolute_error(y_test, y_pred)
test_r2 = r2_score(y_test, y_pred)

print("Linear Regression - TEST metrics")
print(f"RMSE: {test_rmse:.4f}")
print(f"MAE : {test_mae:.4f}")
print(f"R2  : {test_r2:.4f}")

## Residual analysis and error distribution

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

axes[0].scatter(y_test, y_pred, alpha=0.5)
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, "r--")
axes[0].set_xlabel("Actual calories (kcal)")
axes[0].set_ylabel("Predicted calories (kcal)")
axes[0].set_title("Predicted vs actual")

axes[1].scatter(y_pred, residuals, alpha=0.5)
axes[1].axhline(0, color="r", linestyle="--")
axes[1].set_xlabel("Predicted calories (kcal)")
axes[1].set_ylabel("Residual (kcal)")
axes[1].set_title("Residuals vs predicted")

sns.histplot(residuals, kde=True, ax=axes[2])
axes[2].set_xlabel("Residual (kcal)")
axes[2].set_title("Error distribution")

plt.tight_layout()
plt.show()

print(f"Mean residual: {residuals.mean():.2f} kcal | Std: {residuals.std():.2f} kcal")
print(f"Largest absolute error: {residuals.abs().max():.2f} kcal")

**Finding:** _(are residuals centered on zero and evenly spread? any pattern or heteroscedasticity? where are the largest errors?)_

## Coefficients (interpretation)

In [ ]:
coefs = pd.Series(final_pipe.named_steps["model"].coef_, index=FEATURES)
print("Coefficients per 1 SD of each (standardized) feature, in kcal:")
print(coefs.sort_values(ascending=False))

## Save the full pipeline (preprocessing + model) for the Streamlit app

In [ ]:
MODEL_PATH.parent.mkdir(exist_ok=True)
joblib.dump(final_pipe, MODEL_PATH)

reloaded = joblib.load(MODEL_PATH)
assert np.allclose(reloaded.predict(X_test), y_pred)
print("Saved and verified:", MODEL_PATH)